# 01 — Data Simulation

Generate a Murex-like input layer for risk analytics: synthetic positions, portfolio hierarchy, historical market data, and enriched positions ready for pricing.

**Outputs**
- `data/processed/positions_enriched.csv`
- in-memory `positions`, `structure`, `market`, and `enriched` DataFrames


## 1. Notebook setup

The notebook can run from either the repository root or the `notebooks/` folder. It adds the project root to `sys.path` so imports resolve like production code.


In [3]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.6f}".format)

DATA_DIR = PROJECT_ROOT / "data" / "processed"
DATA_DIR.mkdir(parents=True, exist_ok=True)


## 2. Configuration

Use the same `RiskSettings` dataclass as the CLI pipeline. This keeps notebooks aligned with production defaults.


In [4]:
from src.config.settings import RiskSettings

settings = RiskSettings(
    valuation_date="2025-04-04",
    confidence_level=0.99,
    lookback_days=252,
    random_seed=42,
)
settings


RiskSettings(valuation_date='2025-04-04', confidence_level=0.99, lookback_days=252, stress_window_days=252, random_seed=42, base_currency='USD', raw_data_dir=WindowsPath('data/raw'), processed_data_dir=WindowsPath('data/processed'), tickers=('AAPL', 'GOOG', 'EURUSD=X', 'GBPUSD=X'), risk_free_rate=0.02, dividend_yield=0.0, ticker_vols=None)

## 3. Generate positions and hierarchy

Positions represent a compact Murex-style trade extract with stocks, FX forwards, and listed-style European options. The hierarchy maps portfolios to desks and business units for downstream aggregation.


In [5]:
from src.data.generate_positions import generate_synthetic_positions
from src.data.generate_structure import generate_structure

positions = generate_synthetic_positions(settings.valuation_date)
structure = generate_structure()

display(positions)
display(structure)


,PositionID,InstrumentType,Ticker,Quantity,Portfolio,Maturity,Strike,OptionType
0,POS001,Stock,AAPL,1000,P1_EqUS,NaT,NaN,NaN
1,POS002,Stock,GOOG,500,P1_EqUS,NaT,NaN,NaN
2,POS003,FX Forward,EURUSD=X,1000000,P2_FXMaj,2025-07-03,1.080000,NaN
3,POS004,FX Forward,GBPUSD=X,-500000,P2_FXMaj,2025-10-01,1.250000,NaN
4,POS005,European Option,AAPL,50,P3_OptEq,2025-06-03,170.000000,Call
5,POS006,European Option,GOOG,-30,P3_OptEq,2025-08-02,180.000000,Put
6,POS007,Stock,GOOG,-200,P1_EqUS,NaT,NaN,NaN
7,POS008,European Option,AAPL,100,P4_OptSpec,2025-07-03,175.000000,Call
8,POS009,FX Forward,EURUSD=X,-200000,P2_FXMaj,2025-05-04,1.070000,NaN
9,POS010,Stock,AAPL,400,P4_OptSpec,NaT,NaN,NaN


,Portfolio,TradingDesk,Unit
0,P1_EqUS,Equity Desk,Trading Unit A
1,P2_FXMaj,FX Desk,Trading Unit A
2,P3_OptEq,Options Desk,Trading Unit B
3,P4_OptSpec,Options Desk,Trading Unit B


## 4. Load or simulate market data

`load_market_data` can use `yfinance` when enabled, but defaults to deterministic simulated prices so CI and offline execution remain stable.


In [ ]:
from src.data.market_data import load_market_data

market = load_market_data(
    settings.tickers,
    settings.valuation_date,
    seed=settings.random_seed,
    use_yfinance=False,
)

market.tail()


## 5. Enrich positions with hierarchy and latest prices

This creates the canonical pricing input: each position now has desk/unit metadata and the current underlying price.


In [ ]:
from src.data.market_data import enrich_positions_with_market

enriched = enrich_positions_with_market(
    positions=positions,
    structure=structure,
    market=market,
    valuation_date=settings.valuation_date,
)

enriched


## 6. Data quality checks

Production-oriented checks: missing hierarchy, missing prices, unsupported instruments, and duplicate IDs should be caught before pricing.


In [ ]:
assert enriched["PositionID"].is_unique, "PositionID must be unique"
assert enriched["TradingDesk"].notna().all(), "Missing trading desk mapping"
assert enriched["Unit"].notna().all(), "Missing unit mapping"
assert enriched["CurrentPrice"].notna().all(), "Missing market price"
assert set(enriched["InstrumentType"]).issubset({"Stock", "FX Forward", "European Option"})

enriched.groupby(["Unit", "TradingDesk", "InstrumentType"]).size().rename("PositionCount").reset_index()


## 7. Persist enriched positions


In [ ]:
output_path = DATA_DIR / "positions_enriched.csv"
enriched.to_csv(output_path, index=False)
print(f"Wrote {output_path.relative_to(PROJECT_ROOT)}")


## 8. Quick visualization

A simple line chart helps validate that generated paths are sensible before they drive Historical Simulation VaR.


In [ ]:
import matplotlib.pyplot as plt

ax = market[list(settings.tickers)].plot(figsize=(11, 5), title="Historical Market Data")
ax.set_xlabel("Date")
ax.set_ylabel("Price / FX Rate")
plt.tight_layout()
